<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/flagships/humanoid-lab/lessons/F15-L07-sampling-mpc-in-cpp/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/flagships/humanoid-lab/lessons/F15-L07-sampling-mpc-in-cpp/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=flagships/humanoid-lab/lessons/F15-L07-sampling-mpc-in-cpp/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have, and a C or C++ compiler (`clang` or `gcc`). The cell
below installs `mujoco==3.13.0` and fetches the files it needs beside it, and does nothing
where they are already present. On Kaggle, switch Internet on in the notebook's settings
first; Kaggle allows that only for phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("mujoco", "mujoco==3.13.0")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["Makefile", "lesson.cpp", "assets/SOURCE.md", "assets/cartpole.xml"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/flagships/humanoid-lab/lessons/F15-L07-sampling-mpc-in-cpp/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# F15-L07 · Sampling-based MPC, in C++

**You will build:** a Model Predictive Path Integral planner that swings a cart-pole up from
hanging — the sampling, the scoring and the refit written in C++ — plus the measurement that
says whether it could run on a real 100 Hz control loop.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, no GPU, no download
· **Prerequisites:** F15-L01 (mjModel vs mjData), F15-L03 (PD control), F15-L06 (the C loop)

By the end you will be able to:
1. Implement the three stages of an MPPI planner in C++ — sample, score, refit — and pass
   the binary's own self-test.
2. Measure this machine's rollout throughput in C++ and in Python and report the ratio.
3. Compute the sample-and-horizon budget a 100 Hz control tick affords at a measured
   throughput, and decide whether a configuration is real-time.
4. Explain why subtracting the minimum cost before exponentiating is what stops the MPPI
   weights underflowing to zero.
5. Show that scoring a candidate plan leaves the live simulation untouched, and say what
   breaks when it does not.

Everything you implement in this lesson lives in **`lesson.cpp`**. This notebook builds it,
drives it, checks it and measures it. Two small exercises at the end are in Python, and they
exist to let you feel the difference the language makes.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import math
import os
import random
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Callable

import mujoco
import numpy as np

print("mujoco", mujoco.__version__, "· numpy", np.__version__, "· python",
      sys.version.split()[0])

# True in a notebook and when this file is run as a script; False when the autograder
# imports it. Each exercise check below is called under this guard, so the cell you are
# sitting in reports on itself, while importing the lesson never runs anything.
_IS_MAIN = __name__ == "__main__"

# This file is the source of truth for the student build; the reference build lives beside it
# in the reference tree and is compiled from its own .cpp. Detecting which one is running keeps the
# two files identical apart from the exercise bodies.
try:
    _HERE = Path(__file__).resolve().parent
except NameError:  # a notebook has no __file__
    _HERE = Path.cwd()
if _HERE.name == "solutions":
    LESSON_DIR, CPP_SRC, BIN = _HERE.parent, "solutions/lesson.cpp", "solutions/lesson_bin"
else:
    LESSON_DIR, CPP_SRC, BIN = _HERE, "lesson.cpp", "lesson_bin"
MODEL_REL = "assets/cartpole.xml"
MODEL_PATH = LESSON_DIR / MODEL_REL

# The objective, mirrored exactly from lesson.cpp. Both languages must optimise the same
# thing or no comparison between them means anything.
W_ANGLE, W_CART, W_THETADOT, W_XDOT, W_CTRL = 10.0, 1.0, 0.05, 0.05, 0.01

_BUILD = None   # (the source's modification time it was built from, ok, compiler output)


def build(verbose: bool = True):
    """Compile the C++ source with make. Cached, but keyed on the source's modification time:
    save an edit to lesson.cpp and the next check cell you run rebuilds before it runs, so
    "fill it in, then re-run this cell" is literally all it takes."""
    global _BUILD
    try:
        stamp = (LESSON_DIR / CPP_SRC).stat().st_mtime_ns
    except OSError:  # a missing source: let make report it, once
        stamp = None
    if _BUILD is None or _BUILD[0] != stamp:
        # After an edit, -B forces the compile: make compares whole seconds on some systems,
        # and a save in the same second as the last build would otherwise look up to date.
        proc = subprocess.run(
            ["make", "-C", str(LESSON_DIR), *(["-B"] if _BUILD is not None else []),
             f"PYTHON={sys.executable}", f"SRC={CPP_SRC}", f"BIN={BIN}"],
            capture_output=True, text=True, timeout=600)
        _BUILD = (stamp, proc.returncode == 0, (proc.stdout + proc.stderr).strip())
    _, ok, out = _BUILD
    if verbose:
        print("build OK" if ok else "BUILD FAILED\n" + out)
    return ok, out


def run_cpp(*args, timeout: int = 900) -> str:
    """Run the compiled binary and return its stdout.

    Exit code 2 means one of the three C++ exercises is still a stub, so it is re-raised as
    NotImplementedError — the grader then reports TODO instead of an error.
    """
    ok, out = build(verbose=False)
    if not ok:
        raise RuntimeError("the C++ build failed; run build() to see the compiler output\n" + out)
    proc = subprocess.run([str(LESSON_DIR / BIN), *args, "--model", MODEL_REL],
                          cwd=LESSON_DIR, capture_output=True, text=True, timeout=timeout)
    if proc.returncode == 2:
        raise NotImplementedError(proc.stderr.strip())
    if proc.returncode != 0:
        raise RuntimeError(f"{BIN} {' '.join(args)} exited {proc.returncode}\n{proc.stderr.strip()}")
    return proc.stdout


def metrics(text: str) -> dict:
    """Parse the binary's `@ key=value` report lines into a dict of floats."""
    return {k: float(v) for k, v in
            (line[2:].split("=", 1) for line in text.splitlines() if line.startswith("@ "))}


def rows(text: str, tag: str) -> list:
    """Parse the binary's `<tag> v1 v2 ...` lines into a list of lists of floats."""
    return [[float(v) for v in line.split()[1:]]
            for line in text.splitlines() if line.startswith(tag + " ")]


def make_test():
    """Run `make test` — the C++ self-test. Returns (exit code, combined output)."""
    proc = subprocess.run(
        ["make", "-C", str(LESSON_DIR), f"PYTHON={sys.executable}",
         f"SRC={CPP_SRC}", f"BIN={BIN}", "test"],
        capture_output=True, text=True, timeout=900)
    return proc.returncode, (proc.stdout + proc.stderr).strip()


def fresh():
    """A (model, data, scratch) triple reset to the model's `hanging` keyframe."""
    model = mujoco.MjModel.from_xml_path(str(MODEL_PATH))
    data, scratch = mujoco.MjData(model), mujoco.MjData(model)
    mujoco.mj_resetDataKeyframe(model, data, 0)
    mujoco.mj_forward(model, data)
    return model, data, scratch


def stage_cost(scratch, u: float) -> float:
    """The per-step objective, identical to stage_cost() in lesson.cpp."""
    x, th = float(scratch.qpos[0]), float(scratch.qpos[1])
    xd, thd = float(scratch.qvel[0]), float(scratch.qvel[1])
    return (W_ANGLE * (1.0 - math.cos(th)) + W_CART * x * x
            + W_THETADOT * thd * thd + W_XDOT * xd * xd + W_CTRL * u * u)


# The five exercises, in the order the progress board in the last cell lists them.
_EXERCISES = {
    "exercise 1": "sample_controls, in lesson.cpp",
    "exercise 2": "rollout_cost, in lesson.cpp",
    "exercise 3": "refit_mean, in lesson.cpp",
    "exercise 4": "python_rollout_cost, in this notebook",
    "exercise 5": "rollout_budget, in this notebook",
}
# label -> "passed" | "failed" | "not started": the latest verdict of every check that has
# run. The progress board in the last cell reads it.
_STATUS: dict = {}


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. Nothing is swallowed: every verdict is
    recorded in `_STATUS`, and the last cell exits non-zero if any check came back wrong when
    this file runs as a script.

    `needs` names the exercises a demo consumes. Until each of them has passed its own check,
    the demo says which one it is waiting for and skips, instead of failing on its behalf.
    """
    waiting = [n for n in needs if _STATUS.get(n) != "passed"]
    if waiting:
        _STATUS.pop(label, None)
        one = len(waiting) == 1
        names = [f"{n} ({_EXERCISES[n]})" for n in waiting] if len(waiting) <= 2 else waiting
        names = names[0] if one else ", ".join(names[:-1]) + " and " + names[-1]
        print(f"{label}: skipped — it needs {names} to pass first. Finish "
              f"{'that' if one else 'those'}, re-run "
              + ("its check cell" if one else "their check cells") + ", then re-run this one.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        # A C++ stub names itself on the way out; a Python stub raises with no message.
        stub = str(exc).split(" — ")[0].replace("NOT IMPLEMENTED: ", "").strip()
        if stub:
            print(f"{label}: not implemented yet — lesson.cpp reports that {stub}. Fill it "
                  "in, then re-run this cell.")
        else:
            print(f"{label}: not implemented yet — fill in the stub above, then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


build()

## 1. The problem a feedback law cannot solve

The cart-pole starts *hanging*: pole straight down, cart centred, at rest. The goal is the
pole upright and the cart back at the origin.

In F15-L03 you built a PD controller. Point one at this problem and watch what happens. The
code below is given — run it.

In [ ]:
def pd_baseline(kp: float = 30.0, kd: float = 3.0, ticks: int = 400) -> dict:
    """Drive the cart with proportional-derivative feedback on the upright pole angle."""
    model, data, _ = fresh()
    lo, hi = model.actuator_ctrlrange[0]
    start_error = math.atan2(math.sin(data.qpos[1]), math.cos(data.qpos[1]))
    start_command = -kp * start_error - kd * float(data.qvel[1])
    saturated, window = 0, []
    for _ in range(ticks):
        err = math.atan2(math.sin(data.qpos[1]), math.cos(data.qpos[1]))  # wrapped to (-pi, pi]
        want = -kp * err - kd * float(data.qvel[1])
        data.ctrl[0] = float(np.clip(want, lo, hi))
        saturated += int(abs(want) > float(hi))
        mujoco.mj_step(model, data)
        window.append(abs(math.atan2(math.sin(data.qpos[1]), math.cos(data.qpos[1]))))
    return {"final_abs_theta": window[-1],
            "upright_fraction": sum(1 for a in window[-100:] if a < 0.2) / 100,
            "start_error": start_error, "start_command": start_command,
            "ctrl_limit": float(hi), "saturated_fraction": saturated / ticks,
            "seconds": ticks * float(model.opt.timestep)}


_pd = pd_baseline()
print(f"hanging start: the pole is {_pd['start_error']:.3f} rad from upright, so the PD law "
      f"asks for {_pd['start_command']:+.2f} of control")
print(f"the actuator accepts +/-{_pd['ctrl_limit']:.0f}, so it saturates at once — and stays "
      f"saturated for {_pd['saturated_fraction']:.0%} of the run")
print(f"after {_pd['seconds']:.0f} s of PD: final |theta| = {_pd['final_abs_theta']:.3f} rad, "
      f"upright for {_pd['upright_fraction']:.2f} of the last second")

The law is not idle — it pins the actuator to its limit for most of the run — and it still
does not hold the pole up. That is the interesting part. A PD law is a *local* stabiliser: a
linearisation about upright, asked here to work half a revolution away from the point it was
linearised at. Swinging up needs the cart driven *away* from where the law wants it, more
than once, to pump energy in over several swings; a law with no representation of the future
cannot choose to do that. The actuator is not the limitation, as the planner you are about
to write will demonstrate on exactly the same one.

A sampling planner has no such problem, because it does not reason about the objective at
all. It guesses thousands of plans, simulates each one, and keeps what scored well. That is
the entire idea, and it is why a real stack spends its CPU on rollouts — MuJoCo MPC ships
exactly this kind of derivative-free planner alongside its gradient-based ones
(`claims.yaml`, `mjpc-real-time-sampling`).

## 2. What one rollout costs

Before writing a planner, look at the machine you are planning on.

In [ ]:
_model, _data, _scratch = fresh()
print(f"nq={_model.nq}  nv={_model.nv}  nu={_model.nu}  keyframes={_model.nkey}")
print(f"timestep = {_model.opt.timestep} s  ->  control rate = {1 / _model.opt.timestep:.0f} Hz")
print(f"ctrlrange = {_model.actuator_ctrlrange[0]}")
print(f"hanging keyframe: qpos = {np.array(_data.qpos)}  (hinge = pi is straight down)")
print(f"\na horizon of 40 steps is {40 * _model.opt.timestep:.2f} s of predicted motion;")
print(f"200 samples of that horizon is {200 * 40:,} mj_step calls for ONE control tick.")

## 3. The loop you are about to write

Open **`lesson.cpp`**. Three functions are stubs, and together they are the planner:

1. `sample_controls` — draw `samples` candidate plans around the current plan.
2. `rollout_cost` — simulate one plan on a *copy* of the state and score it.
3. `refit_mean` — fold the scores back into a new plan.

The harness around them already works: it applies only the first control of each refitted
plan, steps the real simulation once, shifts the plan along, and repeats.

Your feedback loop is `make test`. Run it now — it should report three TODOs.

In [ ]:
_code, _out = make_test()
print(_out)
print(f"exit code {_code}  (0 = all pass, 2 = something is still a stub, 1 = a real failure)")

## 4. Exercise 1 — `sample_controls` in `lesson.cpp`

Fill `out` with `samples` plans of `horizon` controls, row-major, each one the current plan
plus fresh Gaussian noise, clamped into the actuator's range:

`out[i*horizon + t] = clamp(mean[t] + sigma * z, lo, hi)` with a new `z ~ Normal(0,1)` every
time. The full brief, including the worked example, is in the comment above the function.

The check below pulls the buffer back out of the binary and tests its statistics from here.

<details><summary>💡 Hint 1 — what to think about</summary>

Picture `out` as a table: one row per candidate plan, one column per time step, stored row
after row. Three decisions make or break it — where entry (i, t) lives, where each random
number comes from, and whether the clamp sees the whole value or only the noise. The check
catches a transposed buffer, a reused draw, and a clamp applied too early.

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

Size the buffer first. Loop over samples on the outside and time steps inside, and in the
inner loop draw one new standard normal from the generator you were handed — never a fresh
generator, or the same seed stops replaying. Scale it by sigma, add the current plan's value
at that step, and only then clamp the sum. If you build the distribution with sigma as its
spread, do not multiply by sigma a second time.

</details>

In [ ]:
def _check_sampling() -> None:
    text = run_cpp("sampledump", "--samples", "600", "--horizon", "6",
                   "--sigma", "0.35", "--offset", "-0.2", "--seed", "99")
    head, block = metrics(text), np.array(rows(text, "row"))
    assert block.shape == (600, 6), (
        f"sampledump returned {block.shape}, expected (600, 6) — `out` must hold "
        "samples*horizon values, not one plan")
    assert block.min() >= head["lo"] - 1e-12 and block.max() <= head["hi"] + 1e-12, (
        f"values ran from {block.min():.3f} to {block.max():.3f} but ctrlrange is "
        f"[{head['lo']}, {head['hi']}] — clamp after adding the noise")
    assert not np.allclose(block[0], block[1]), (
        "the first two plans are identical — you reused one noise draw across samples")
    assert abs(float(block.mean()) + 0.2) < 0.04, (
        f"the sample mean is {block.mean():.4f} but the plan being perturbed was -0.2 — "
        "add mean[t] to the noise")
    assert abs(float(block.std()) - 0.35) < 0.04, (
        f"the spread is {block.std():.4f} against sigma = 0.35 — multiply a *standard* normal "
        "by sigma; normal_distribution's second argument is a standard deviation")
    again = np.array(rows(run_cpp("sampledump", "--samples", "600", "--horizon", "6",
                                  "--sigma", "0.35", "--offset", "-0.2", "--seed", "99"), "row"))
    assert np.array_equal(block, again), (
        "the same seed gave a different buffer — draw from the rng you were handed")
    print(f"exercise 1 looks right: 600x6 samples, mean {block.mean():+.4f}, "
          f"sd {block.std():.4f}, all within [{head['lo']:.0f}, {head['hi']:.0f}]")


if _IS_MAIN:
    _try("exercise 1", _check_sampling)

## 5. Exercise 2 — `rollout_cost` in `lesson.cpp`

Branch `scratch` off `start` with `load_state`, then for each `t`: write `controls[t]` into
`scratch->ctrl[0]`, call `mj_step` once, and add `stage_cost(scratch, controls[t])`.

`start` is `const`. Step it instead of the copy and the planner's hypothetical futures
become real motion — the robot is then driven by its own imagination, and no two samples
start from the same state, so their costs cannot be compared.

<details><summary>💡 Hint 1 — what to think about</summary>

Two copies of the world are in play: `start`, the robot as it really is, and `scratch`, a
whiteboard you may scribble on. For every line you write, ask which one it touches. The
check scores one plan twice and compares, then reads the live hinge angle and clock.

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

Branch `scratch` off `start` at the top of EVERY call, before the loop — not once per
program, or the second call begins where the first one ended. Then, per step and in this
order: write the control, advance the copy by one step, and score the state the copy has
just arrived at with that same control. Return the running total, not the last stage.

</details>

In [ ]:
def _check_rollout() -> None:
    rng = random.Random(20260916)
    plan = [max(-1.0, min(1.0, rng.gauss(0.0, 0.6))) for _ in range(30)]
    # The scratch file lives outside the lesson directory on purpose: anything written under
    # assets/ would be swept into the student bundle if this check were interrupted mid-run.
    fd, ctrl_path = tempfile.mkstemp(suffix=".txt", prefix="f15l07_check_")
    try:
        with os.fdopen(fd, "w") as fh:
            fh.write("\n".join(repr(v) for v in plan))
        first = metrics(run_cpp("costof", "--controls", ctrl_path))
        second = metrics(run_cpp("costof", "--controls", ctrl_path))
        assert first["cost"] > 0.0 and math.isfinite(first["cost"]), (
            f"the rollout scored {first['cost']} — every weight is non-negative and the pole "
            "starts hanging, so the total must be positive and finite")
        assert first["cost"] == second["cost"], (
            "scoring the same plan twice gave different answers — scratch state is leaking "
            "between calls; branch it from `start` at the top of every call")
        assert abs(first["start_theta_after"] - math.pi) < 1e-12, (
            f"after scoring, the live state's hinge is {first['start_theta_after']} but it "
            "started at pi — you stepped `start` instead of `scratch`")
        assert first["start_time_after"] == 0.0, (
            "the live simulation clock advanced during scoring — again, you are stepping the "
            "real mjData rather than the copy")

        Path(ctrl_path).write_text("\n".join(repr(v) for v in plan * 2))
        longer = metrics(run_cpp("costof", "--controls", ctrl_path))
        assert longer["cost"] > first["cost"], (
            "doubling the horizon did not raise the cost — you are returning the last stage's "
            "cost rather than the sum over the horizon")
        print(f"exercise 2 looks right: 30 steps score {first['cost']:.3f}, 60 steps "
              f"{longer['cost']:.3f}, and the live state never moved")
    finally:
        os.unlink(ctrl_path)


if _IS_MAIN:
    _try("exercise 2", _check_rollout)

## 6. Exercise 3 — `refit_mean` in `lesson.cpp`

Turn costs into weights and average the plans under them:

`w_i = exp(-(cost_i - min_j cost_j) / lambda)`, then `mean[t] = sum_i (w_i / sum_j w_j) * u_i[t]`.

This is the MPPI update (Williams, Aldrich and Theodorou, 2015 — see `claims.yaml`). The
subtraction of the minimum cost is the part everyone omits and it is the part that matters:
it changes nothing mathematically, because the constant cancels between the numerator and
the denominator, but without it `exp(-30000/30)` is exactly `0`, every weight vanishes, and
you divide zero by zero.

The alternative refit is the cross-entropy one: keep the best-scoring *elite* fraction and
average only those. MuJoCo MPC ships that planner too (`claims.yaml`,
`mjpc-cross-entropy-planner`). Softmax is a soft version of the same instinct.

<details><summary>💡 Hint 1 — what to think about</summary>

The check tries three limits of one formula: equal costs must give the plain average, one
outrageously cheap plan must give that plan, and adding one huge constant to every cost must
change nothing at all. Work out what the weights are in each case before you write the loop
— the third is where the textbook formula turns into NaN.

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

Two passes. The first finds the cheapest cost and weighs each sample by how far its cost
sits ABOVE the cheapest, divided by lambda, negated inside the exponential — so the best
plan always weighs exactly one and the total can never be zero. The second clears the plan
and adds each sample's control scaled by its weight over the total weight. Keep lambda in
the exponent: it sets how sharply the refit favours the winners.

</details>

In [ ]:
def _check_refit() -> None:
    equal = run_cpp("refitdump", "--samples", "8", "--horizon", "5", "--costs", "equal")
    mean_eq, arith = rows(equal, "mean")[0], rows(equal, "arithmetic")[0]
    assert np.allclose(mean_eq, arith, atol=1e-12), (
        "with every cost equal every weight is equal, so the refit must be the plain "
        "arithmetic mean of the samples — check that you divide by the sum of the weights")

    spike = run_cpp("refitdump", "--samples", "8", "--horizon", "5", "--costs", "spike")
    assert np.allclose(rows(spike, "mean")[0], rows(spike, "first")[0], atol=1e-6), (
        "one sample was astronomically cheaper than the rest, so the refit should be that "
        "sample almost exactly — your weights are not falling off with cost")

    small = rows(run_cpp("refitdump", "--samples", "8", "--horizon", "5",
                         "--costs", "graded", "--offset", "0"), "mean")[0]
    big = rows(run_cpp("refitdump", "--samples", "8", "--horizon", "5",
                       "--costs", "graded", "--offset", "1e6"), "mean")[0]
    assert all(math.isfinite(v) for v in big), (
        "shifting every cost up by 1e6 produced a non-finite plan — exp(-1e6/lambda) "
        "underflows to exactly 0, so every weight vanished and you divided by zero")
    assert np.allclose(small, big, atol=1e-12), (
        "adding the same constant to every cost changed the answer — it must cancel; "
        "subtract the minimum cost before exponentiating")
    print(f"exercise 3 looks right: equal costs give the plain mean, a cost spike gives the "
          f"winner, and a +1e6 shift moves the plan by "
          f"{max(abs(a - b) for a, b in zip(small, big)):.1e}")


if _IS_MAIN:
    _try("exercise 3", _check_refit)

## 7. Exercise 4 — the same rollout, in Python

Now write `rollout_cost` again, here, in Python. Same objective, same model, same start —
the only thing that changes is the language. This is what makes the comparison later an
apples-to-apples one rather than a slogan.

<details><summary>💡 Hint 1 — what to think about</summary>

This is exercise 2 again, so the C++ you wrote is the specification, and the check holds
you to it to rounding error: any difference in the ORDER of control, step and score shows.
It also checks the type you return, and that `data` never moved.

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

Copy `data` into `scratch` first — that is what starts the rollout from the hanging
keyframe rather than the model's default pose. Then for each control: set it, step the
copy, add the stage cost of where the copy now is. Start the total at a plain Python zero
point zero and you never hand back a numpy scalar.

</details>

In [ ]:
def python_rollout_cost(model, data, scratch, controls) -> float:
    """Score one plan in Python, exactly as rollout_cost() does in lesson.cpp.

    Branch `scratch` off `data` with `mujoco.mj_copyData(scratch, model, data)`, then for each
    `u` in `controls`: set `scratch.ctrl[0] = u`, call `mujoco.mj_step(model, scratch)`, and
    add `stage_cost(scratch, u)` to a running total. Leave `data` untouched.

    Example (longer plans cost more, because every weight is non-negative):
        >>> model, data, scratch = fresh()
        >>> two = python_rollout_cost(model, data, scratch, [0.0, 0.0])
        >>> one = python_rollout_cost(model, data, scratch, [0.0])
        >>> two > one > 0
        True

    Returns:
        A single float — the summed stage cost over the plan. A plain Python float, not a
        numpy scalar: start `total` at 0.0 and it stays one, or wrap the result in float().
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_python_rollout() -> None:
    rng = random.Random(7771)
    plan = [max(-1.0, min(1.0, rng.gauss(0.0, 0.6))) for _ in range(40)]
    model, data, scratch = fresh()
    mine = python_rollout_cost(model, data, scratch, plan)
    assert type(mine) is float, (
        f"python_rollout_cost returned {type(mine).__name__} — sum into a plain Python float "
        "and return it. A numpy scalar subclasses float, so isinstance() would not catch it; "
        "float(total) is the fix")
    assert abs(float(data.qpos[1]) - math.pi) < 1e-12, (
        "the live `data` moved — you stepped it instead of `scratch`; copy first")

    fd, ctrl_path = tempfile.mkstemp(suffix=".txt", prefix="f15l07_py_")
    try:
        with os.fdopen(fd, "w") as fh:
            fh.write("\n".join(repr(v) for v in plan))
        theirs = metrics(run_cpp("costof", "--controls", ctrl_path))["cost"]
    finally:
        os.unlink(ctrl_path)
    rel = abs(mine - theirs) / abs(theirs)
    assert rel < 1e-9, (
        f"Python scored {mine!r} and C++ scored {theirs!r} (relative gap {rel:.2e}) — the two "
        "loops must be the same computation. Check that you step AFTER setting ctrl, that you "
        "score the state after the step, and that you start from the keyframe, not qpos0")
    print(f"exercise 4 looks right: Python {mine:.9f} vs C++ {theirs:.9f}, "
          f"relative gap {rel:.1e}")


# It compares your Python with your C++ rollout_cost, so it waits for exercise 2.
if _IS_MAIN:
    _try("exercise 4", _check_python_rollout, needs=("exercise 2",))

## 8. Exercise 5 — what the control tick can afford

A planner is real-time only if one planning tick fits inside one control period. At 100 Hz
that is 10 ms to spend on `samples * horizon` calls to `mj_step`. Write the arithmetic that
turns a measured throughput into a verdict.

<details><summary>💡 Hint 1 — what to think about</summary>

Every key is a unit conversion. Write the unit beside each — steps, steps per second,
seconds — and the divisions choose their own direction. Then look at the two keys that
count whole things: the check asks for their TYPE as well as their value, and it tries a
throughput whose affordable sample count is not a whole number.

</details>

<details><summary>💡 Hint 2 — the approach, in words</summary>

Work per tick is samples times horizon; the time that work takes is work over throughput;
the tick is one over the control rate. Real-time means the first is no longer than the
second, and a plan that exactly fills the tick counts. The affordable sample count is the
throughput over what one sample costs per second — rate times horizon — rounded DOWN, as an
int. Let a zero throughput raise; do not catch it.

</details>

In [ ]:
def rollout_budget(samples: int, horizon: int, control_hz: float,
                   steps_per_second: float) -> dict:
    """Decide whether a planner configuration fits inside one control tick.

    Example:
        >>> b = rollout_budget(200, 40, 100.0, 1_000_000.0)
        >>> b["steps_per_tick"], b["steps_per_second_required"]
        (8000, 800000.0)
        >>> b["tick_seconds"], b["plan_seconds_per_tick"]
        (0.01, 0.008)
        >>> b["realtime"], b["max_samples"]
        (True, 250)

    Returns:
        dict with exactly these six keys:
          "steps_per_tick"            int, samples * horizon
          "steps_per_second_required" float, steps_per_tick * control_hz
          "tick_seconds"              float, 1 / control_hz
          "plan_seconds_per_tick"     float, steps_per_tick / steps_per_second
          "realtime"                  bool, True when plan_seconds_per_tick <= tick_seconds.
                                      Planning that exactly fills the tick still counts as
                                      fitting, so compare with <=, not <.
          "max_samples"               int, the largest WHOLE number of samples that still
                                      fits: floor(steps_per_second / (control_hz * horizon)).
                                      Round DOWN — 308.6 affordable samples means 308.

    `steps_per_tick` and `max_samples` count whole things, so return them as `int`. A float
    that happens to compare equal (8000.0 == 8000) is not the same answer.

    Do not guard the arithmetic with `try`/`except`. If the throughput is zero there is no
    budget to report, and a zeroed dict would read exactly like one.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_budget() -> None:
    b = rollout_budget(200, 40, 100.0, 1_000_000.0)
    expected = {"steps_per_tick", "steps_per_second_required", "tick_seconds",
                "plan_seconds_per_tick", "realtime", "max_samples"}
    assert set(b) == expected, (
        f"keys were {sorted(b)}, expected {sorted(expected)} — spell them as in the docstring")
    assert b["steps_per_tick"] == 8000, (
        f"steps_per_tick was {b['steps_per_tick']}, expected 200*40 = 8000")
    assert b["plan_seconds_per_tick"] == 0.008, (
        f"plan_seconds_per_tick was {b['plan_seconds_per_tick']}, expected 8000/1e6 = 0.008 — "
        "divide the work by the throughput, not the other way round")
    assert b["realtime"] is True, (
        "8 ms of planning fits inside a 10 ms tick, so realtime must be True — compare "
        "plan_seconds_per_tick against tick_seconds")
    assert type(b["steps_per_tick"]) is int and type(b["max_samples"]) is int, (
        f"steps_per_tick came back as {type(b['steps_per_tick']).__name__} and max_samples as "
        f"{type(b['max_samples']).__name__} — both count whole things, so both are ints. "
        "8000.0 == 8000 is True, so the equality checks above cannot warn you about this")
    try:
        _bad = rollout_budget(200, 40, 100.0, 0.0)
    except ZeroDivisionError:
        pass
    else:
        raise AssertionError(
            f"a throughput of 0 returned {_bad} instead of raising — do not wrap the "
            "arithmetic in try/except. A zeroed budget reads like a real one and hides the "
            "bad throughput that produced it")
    assert b["max_samples"] == 250, (
        f"max_samples was {b['max_samples']}, expected 1e6/(100*40) = 250 — floor it to a "
        "whole number of samples")
    slow = rollout_budget(200, 40, 100.0, 300_000.0)
    assert slow["realtime"] is False, (
        "at 300k steps/s, 8000 steps take 26.7 ms and cannot fit a 10 ms tick — realtime "
        "must be False here")
    assert slow["max_samples"] == 75, (
        f"max_samples was {slow['max_samples']}, expected 300000/(100*40) = 75")
    # The three cases above all divide out exactly, so they cannot tell flooring from
    # rounding. This one can: 1234567 / (100 * 40) = 308.64...
    odd = rollout_budget(200, 40, 100.0, 1_234_567.0)
    assert odd["max_samples"] == 308, (
        f"max_samples was {odd['max_samples']}, expected floor(1234567/(100*40)) = 308 — "
        "308.64 affordable samples is 308 samples; rounding up buys a tick you cannot pay for")
    # A planner that exactly fills the tick still fits.
    exact = rollout_budget(100, 40, 100.0, 400_000.0)
    assert exact["plan_seconds_per_tick"] == exact["tick_seconds"], (
        f"4000 steps at 400k steps/s is exactly one 10 ms tick, but plan_seconds_per_tick "
        f"came out {exact['plan_seconds_per_tick']} against a tick of {exact['tick_seconds']}")
    assert exact["realtime"] is True, (
        "planning that exactly fills the tick still fits — compare with <=, not <")
    print(f"exercise 5 looks right: at 1e6 steps/s a 200x40 planner needs "
          f"{b['plan_seconds_per_tick'] * 1e3:.1f} ms of a "
          f"{b['tick_seconds'] * 1e3:.0f} ms tick; at 3e5 steps/s it needs "
          f"{slow['plan_seconds_per_tick'] * 1e3:.1f} ms")


if _IS_MAIN:
    _try("exercise 5", _check_budget)

## 9. The payoff — swing it up, then measure the language

With all five exercises done, this runs the planner end to end and then benchmarks the same
rollout loop in both languages on identical work. Every number below is produced by the code
you just wrote; none of them are typed into this notebook.

In [ ]:
PY_BENCH_ROLLOUTS, PY_BENCH_HORIZON = 800, 40


def python_throughput(rollouts: int = PY_BENCH_ROLLOUTS, horizon: int = PY_BENCH_HORIZON) -> dict:
    """Time the Python rollout loop on exactly the work `bench` does in C++."""
    model, data, scratch = fresh()
    rng = random.Random(31337)
    plans = [[max(-1.0, min(1.0, rng.gauss(0.0, 0.5))) for _ in range(horizon)]
             for _ in range(rollouts)]
    t0 = time.perf_counter()
    for plan in plans:
        python_rollout_cost(model, data, scratch, plan)
    wall = time.perf_counter() - t0
    steps = rollouts * horizon
    return {"rollouts": rollouts, "horizon": horizon, "steps": steps, "wall_seconds": wall,
            "steps_per_second": steps / wall, "rollouts_per_second": rollouts / wall}


def report() -> None:
    swing = metrics(run_cpp("mpc"))
    print("closed-loop swing-up")
    print(f"  {swing['samples']:.0f} samples x {swing['horizon']:.0f} steps "
          f"({swing['horizon_seconds']:.2f} s of prediction) for {swing['ticks']:.0f} ticks")
    print(f"  {swing['planning_steps']:,.0f} mj_step calls in {swing['wall_seconds']:.2f} s "
          f"wall = {swing['steps_per_second']:,.0f} steps/s")
    print(f"  final |theta| {swing['final_abs_theta']:.4f} rad · upright fraction "
          f"{swing['upright_fraction']:.2f} · cart stayed within "
          f"{swing['max_abs_cart']:.2f} m")
    print(f"  swung up: {'yes' if swing['swung_up'] else 'no'}   "
          f"(PD baseline finished at |theta| = {_pd['final_abs_theta']:.3f} rad)")

    cpp = metrics(run_cpp("bench", "--samples", str(PY_BENCH_ROLLOUTS),
                          "--horizon", str(PY_BENCH_HORIZON)))
    py = python_throughput()
    ratio = cpp["steps_per_second"] / py["steps_per_second"]
    print(f"\nidentical work, two languages: {int(cpp['steps']):,} mj_step calls")
    print(f"  C++    {cpp['wall_seconds'] * 1e3:8.1f} ms  {cpp['steps_per_second']:12,.0f} steps/s"
          f"  {cpp['rollouts_per_second']:9,.0f} rollouts/s")
    print(f"  Python {py['wall_seconds'] * 1e3:8.1f} ms  {py['steps_per_second']:12,.0f} steps/s"
          f"  {py['rollouts_per_second']:9,.0f} rollouts/s")
    # The ratio is a timing, so it is printed beside the two timings it divides: this run's
    # figure, not a constant — and a line that visibly moves from run to run.
    print(f"  C++ is {ratio:.1f}x faster on the same mj_step calls "
          f"({py['wall_seconds'] * 1e3:.1f} ms / {cpp['wall_seconds'] * 1e3:.1f} ms, this run)")

    hz = swing["control_hz"]
    n, h = int(swing["samples"]), int(swing["horizon"])
    print(f"\ncan this planner close a {hz:.0f} Hz loop?  ({n} samples x {h} steps)")
    for name, rate in (("C++", cpp["steps_per_second"]), ("Python", py["steps_per_second"])):
        b = rollout_budget(n, h, hz, rate)
        print(f"  {name:6s} {b['plan_seconds_per_tick'] * 1e3:7.2f} ms per "
              f"{b['tick_seconds'] * 1e3:.0f} ms tick -> "
              f"{'real-time' if b['realtime'] else 'TOO SLOW':9s} "
              f"(affords ≈{b['max_samples']:,} samples at this horizon)")

    print(f"\nthe trade-off, at the C++ rate of {cpp['steps_per_second']:,.0f} steps/s")
    print(f"  {'samples':>8} {'horizon':>8} {'predict':>9} {'plan/tick':>10}  verdict")
    for n_s, h_s in ((100, 20), (200, 40), (500, 40), (1000, 60), (2000, 100)):
        b = rollout_budget(n_s, h_s, hz, cpp["steps_per_second"])
        print(f"  {n_s:8d} {h_s:8d} {h_s / hz:8.2f}s {b['plan_seconds_per_tick'] * 1e3:9.2f}ms  "
              f"{'fits' if b['realtime'] else 'misses the tick'}")

## 10. Common mistakes

- **Stepping `start` instead of `scratch`.** The planner's imagined futures become real
  motion. Symptom: the cost of a plan depends on how many plans you scored before it, and
  the simulation clock races ahead of the control loop.
- **Forgetting to subtract the minimum cost.** Works on toy costs, fails silently on real
  ones: every weight underflows to `0`, the sum is `0`, and the plan becomes `NaN`. Once one
  `NaN` enters the mean it never leaves.
- **One noise draw per sample instead of per (sample, step).** Each candidate plan is then a
  constant offset, the planner can only search over constants, and a swing-up needs a
  control that changes sign.
- **Clamping before adding the mean.** Clamp the final value: it is the actuator that has
  limits, not the noise.
- **Applying the whole plan.** Applying all `horizon` controls before replanning is
  open-loop optimisation, not MPC. Apply the first control, then replan from what actually
  happened.
- **Re-planning from zero every tick.** The shift-by-one warm start is most of why this
  works at 200 samples. Without it you are starting the search over 100 times a second.
- **Hard-coding the timestep.** This model overrides MuJoCo's documented default of 0.002 s
  (`claims.yaml`). Read `model->opt.timestep` and your horizon stays honest.
- **Quoting a speed-up you did not measure.** The ratio between C++ and Python here is a
  property of this model, this machine and this loop. Measure it; do not repeat it.
- **A stale binary after moving or rebuilding your checkout.** The linker bakes the ABSOLUTE
  path of `libmujoco` into `lesson_bin`. Move the repository, rename a parent directory, or
  rebuild the virtualenv somewhere else, and the binary still points at where the library
  used to be: every command dies with `dyld: Library not loaded` (Linux: `error while
  loading shared libraries`) before a line of your code runs. `make` will not rescue you —
  the binary is newer than `lesson.cpp`, so it looks up to date and make does nothing. The
  fix is one line:

      make clean && make

  Run it after any move, rename or virtualenv rebuild. If a command that worked yesterday
  fails today with a library path in the message, this is why — the C++ you wrote is fine.

In [ ]:
# The second mistake, made concrete. There is no C++ here: this is the same exp() your
# refit_mean calls, on costs the size this task actually produces.
def weight_spread(costs, lam: float = 30.0) -> dict:
    """Softmax weights with and without the minimum-cost shift, side by side."""
    best = min(costs)
    return {"unshifted_sum": sum(math.exp(-c / lam) for c in costs),
            "shifted_sum": sum(math.exp(-(c - best) / lam) for c in costs)}


if _IS_MAIN:
    for label, _costs in (("a toy cost", [3.0, 5.0, 8.0]),
                          ("this task", [310.0, 480.0, 900.0]),
                          ("a long horizon", [30000.0, 31000.0, 32000.0])):
        _w = weight_spread(_costs)
        _verdict = "usable" if _w["unshifted_sum"] > 0.0 else "0 -> NaN plan"
        print(f"  {label:<15s} costs {_costs[0]:>8.0f}..{_costs[-1]:<8.0f} "
              f"unshifted sum {_w['unshifted_sum']:.3e} ({_verdict:13s})  "
              f"shifted sum {_w['shifted_sum']:.4f}")
    print("\n  the shifted sum can never be zero: the best sample always weighs exp(0) = 1")

## 11. Self-check

1. From hanging, the PD law saturates the actuator immediately and still fails to hold the
   pole upright. What is the real reason?
   - (a) the gains are too low
   - (b) it is a local stabiliser with no model of the future, so it cannot choose to drive
         the cart the wrong way first and pump energy in over several swings
   - (c) the actuator is too weak to get this pole upright at all
   - (d) contact is disabled in the model

2. What does subtracting `min(cost)` before the exponential do?
   - (a) it makes the planner greedier
   - (b) nothing mathematically — the constant cancels — but it stops every weight
         underflowing to zero
   - (c) it normalises the weights so they sum to one
   - (d) it converts costs into rewards

3. You replace the softmax refit with "average the best 10% of samples". What have you
   built?
   - (a) nothing that works
   - (b) a cross-entropy-style elite refit, a real alternative that MuJoCo MPC also ships
   - (c) gradient descent on the cost
   - (d) the same algorithm, written differently

4. You accidentally step `start` rather than `scratch` inside `rollout_cost`. What is the
   first symptom?
   - (a) the cost goes negative
   - (b) `mj_step` refuses to run on a const pointer
   - (c) scoring the same plan twice gives different answers, because every rollout leaves
         the state somewhere new
   - (d) nothing, MuJoCo copies `mjData` automatically

5. Your benchmark says 1,200,000 steps/s. At 100 Hz with 200 samples, you double the horizon
   from 40 to 80. What happens to the sample count you can afford?
   - (a) it is unchanged
   - (b) it halves
   - (c) it quarters
   - (d) it doubles

Answers, with reasoning, are in this lesson's worked solution in the course repository.

In [ ]:
# The payoff, end to end. Every number this prints is produced by the code you wrote.
# It consumes all five exercises; until each has passed, it names the ones it is waiting for.
if _IS_MAIN:
    _try("the payoff", report, needs=tuple(_EXERCISES))

## What you built, and where it goes next

A complete sampling planner: draw plans, simulate them on copies of the state, weight them
by what they cost, and act on the first control of the result — then throw the rest away and
do it again. You also measured the thing that decides whether it can run at all, which is
how many times a second this machine can call `mj_step`.

That measurement is the systems argument in one number. The algorithm is indifferent to the
language; the control rate is not. The next lesson in the Humanoid Lab takes the same loop
to a model with contact, where each `mj_step` costs far more and the sample budget gets
decided for you.

In [ ]:
# Your progress board. It reads the verdict each check cell recorded the last time it ran, so
# after you fix an exercise, re-run that exercise's check cell and then this one.
def _progress_board() -> list:
    """Print one line per exercise, then the tally. Returns the labels that came back wrong."""
    marks = {"passed": "✅", "failed": "❌", "not started": "⏳"}
    print("\nprogress board")
    for label, what in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {marks[state]} {label}  {what:<38s} {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    return [label for label, state in _STATUS.items() if state == "failed"]


if _IS_MAIN:
    _failed = _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, so a green exit code cannot paper
    # over it. Inside a notebook kernel the same verdict is a printed line, not a traceback.
    if _failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(_failed))
    if _failed:
        print("checks failed: " + ", ".join(_failed) + " — each printed its reason above")